<a href="https://colab.research.google.com/github/jovitaand/The-Effects-of-Stress-in-Gene-Expression-in-Adipose-Tissues/blob/main/Final_The_Effects_of_Stress_in_Gene_Expression_in_Adipose_Tissues.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Problem Statement:
###The effects of Stress in Gene Expression in Adipose Tissues

### What this notebook does

Mice were put through two stress protocols and compared against unstressed controls:

* **CSDS**, chronic social defeat stress
* **CMVS**, chronic multimodal variable stress

Two fat depots were collected from each animal, subcutaneous white fat (**scW**) and brown fat (**BAT**), and profiled by LC-MS. The data file holds peak areas for 301 identified metabolites. The peak area is a stand-in for how much of that metabolite was present.

I run four comparisons, each control against one stress group in one tissue.

For every metabolite I calculate two different measures of "does this matter":

1. a **VIP score** from a PLS-DA model, which is a multivariate method that looks at all metabolites together, and
2. a **p-value** from a Welch t-test, which looks at one metabolite at a time.

The figure plots abundance against consistency and shows VIP as size and colour. The point of it is that the two measures do not agree, and the plot shows why: VIP rewards abundant metabolites, while the t-test does not care how abundant something is.

A note on the title. It says gene expression, but this data is metabolite profiling, so nothing here touches transcripts. The title should probably be changed before this is shared.

### How to run it

Run the cells top to bottom. Set `METAB_XLSX` to the path of the Excel file, or edit `EXCEL_PATH` in the settings cell. The figures are written to `FIG_OUT`, which defaults to the current folder.

## Step 0. Load the tools

Nothing clever happens here. I just import the libraries the rest of the notebook needs.

* `pandas` reads the Excel file and holds the data as a table.
* `numpy` does the maths on arrays of numbers.
* `scipy.stats` gives me the t-test and the t-distribution.
* `matplotlib` draws the figures.

The line `matplotlib.use("Agg")` tells matplotlib not to open a window. It just writes PNG files straight to disk, which is what I want when the notebook runs on a server or in Colab.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

## Step 1. Settings

I keep every setting I might want to change in one place, so I never have to hunt through the code later.

* `EXCEL_PATH` is where the data file lives. `os.environ.get` means I can point at a different file without editing the notebook.
* `SHEET = "Annotation"` is the sheet with the 301 metabolites that were actually identified. The raw sheets have thousands of unnamed features, and I do not want those here.
* `N_COMPONENTS = 2` is how many PLS components the model uses.
* `EXCLUDED_SAMPLE` drops one BAT sample, `BAT_Aq_CMVS6`. It sat far away from everything else in the earlier PCA, so it was treated as an outlier. Nothing is dropped from scW.
* `VIP_COLOUR_CAP` and `VIP_CURVE_LEVELS` control the colour scale and which equal-VIP lines get drawn.
* `ABUNDANT_TOP = 60` is my cut-off for calling a metabolite "abundant". It means the top 60 out of 301 by average peak area.
* The last three numbers are only about spacing in the figure. They stop points and text from touching the edge of the plot.

In [ ]:
EXCEL_PATH = os.environ.get(
    "METAB_XLSX",
    "/content/Yana copy_24April26_scW_BAT_metabolite_profiling_Data_ID.xlsx",
)

OUTPUT_DIR = os.environ.get("FIG_OUT", ".")

SHEET = "Annotation"

N_COMPONENTS = 2

EXCLUDED_SAMPLE = {
    "scW": None,
    "BAT": "BAT_Aq_CMVS6",
}

VIP_COLOUR_CAP = 4.0
VIP_CURVE_LEVELS = [0.5, 1, 3]

ABUNDANT_TOP = 60

# Fraction of the y-range kept below the significance line so that the
# dashed line and its label are actually visible inside the axes.
BOTTOM_MARGIN = 0.04

# Padding (in decades) added to each side of the x-axis so that markers
# and labels never touch or cross the axes frame.
X_PAD_DECADES_LEFT = 0.18
X_PAD_DECADES_RIGHT = 0.35

## Step 2. Reading the data

The Excel file stores one column per injection, and the column name carries all the information about that injection. For example, `Area: scW_Aq_CSDS3` tells me the tissue (scW), the extraction (aqueous) and the group (CSDS).

`describe_injection` reads that name and returns the tissue and the group:

* tissue is `scW` (subcutaneous white fat) or `BAT` (brown fat),
* group is `CSDS`, `CMVS`, or `control` if neither word appears.

Blanks and QC injections are not real samples, so the function returns `None` for them and they get skipped later.

`load_table` opens the sheet with `header=2`, because the first two rows of the sheet are title rows and the real column names are on the third row. It then keeps only the columns starting with `Area: `, which are the peak areas, and forces them to be numbers. Anything that cannot be read as a number becomes `NaN` instead of crashing the run.

In [ ]:
# ==========================================================================
# 1. DATA
# ==========================================================================

def describe_injection(column_name):
    """
    Return (tissue, group) for an injection column,
    or None for QC/blanks.
    """

    if "Blank" in column_name or "QC" in column_name:
        return None

    tissue = "scW" if column_name.startswith("Area: scW") else "BAT"

    if "CMVS" in column_name:
        group = "CMVS"
    elif "CSDS" in column_name:
        group = "CSDS"
    else:
        group = "control"

    return tissue, group


def load_table(excel_path=None, sheet=SHEET):
    """
    Load the metabolomics table and identify area columns.
    """

    excel_path = excel_path or EXCEL_PATH

    table = pd.read_excel(
        excel_path,
        sheet_name=sheet,
        header=2
    )

    area_columns = [
        c for c in table.columns
        if str(c).startswith("Area: ")
    ]

    intensities = table[area_columns].apply(
        pd.to_numeric,
        errors="coerce"
    )

    return table, intensities

## Step 3. The PLS-DA model, built from scratch

This is the heaviest cell, so here is the plain version of what it does.

**Why scale at all?** Peak areas are wildly different in size. Some metabolites come in at a billion, others at ten thousand. If I feed the raw numbers to the model, the big ones drown out everything else just because they are big.

`pareto_scale` fixes some of that. It subtracts the mean of each metabolite, then divides by the square root of the standard deviation. Dividing by the full standard deviation would make every metabolite exactly equally important. The square root only shrinks the gap, so abundant metabolites still count for a bit more. That is the standard choice in metabolomics, and it matters for how the figures look later.

`make_dummy_y` turns the group labels (control, CSDS) into numbers the model can use. Each sample gets a column of 0s and 1s saying which group it belongs to, then the columns are centred and scaled.

`extract_pls_component` finds one PLS component using the NIPALS algorithm. In words: it looks for the combination of metabolites that best lines up with the group labels, then keeps repeating and adjusting until the answer stops changing (that is what the `tolerance` check is for).

`fit_vip` runs that step `n_components` times. After each component it subtracts what has been explained so far and looks again at the leftovers, which is what `x_residual` and `y_residual` are. It then turns the results into **VIP scores**: one number per metabolite saying how much it contributed to separating the groups.

The rule of thumb is that VIP above 1 is worth a second look. That is a convention, not a p-value, and it does not come with any error control.

In [ ]:
# ==========================================================================
# 2. THE MODEL
# ==========================================================================

def pareto_scale(matrix):
    """
    Centre each metabolite, then divide by the square root of its SD.
    """

    mean = matrix.mean(axis=0)

    sd = matrix.std(axis=0, ddof=1)

    sd = np.where(sd == 0, 1.0, sd)

    return (matrix - mean) / np.sqrt(sd)


def make_dummy_y(labels):
    """
    Class labels -> centred, unit-variance dummy response matrix.
    """

    classes = sorted(set(labels))

    response = np.zeros((len(labels), len(classes)))

    for row, label in enumerate(labels):
        response[row, classes.index(label)] = 1.0

    response = response - response.mean(axis=0)

    sd = response.std(axis=0, ddof=1)

    sd = np.where(sd == 0, 1.0, sd)

    return response / sd


def extract_pls_component(
    x_block,
    y_block,
    tolerance=1e-12,
    max_iterations=2000
):
    """
    Extract one PLS2 component using NIPALS.
    """

    start = int(np.argmax((y_block ** 2).sum(axis=0)))

    y_score = y_block[:, [start]].copy()

    x_weight = x_score = y_loading = None

    for _ in range(max_iterations):

        x_weight = x_block.T @ y_score

        norm = np.linalg.norm(x_weight)

        if norm == 0:
            break

        x_weight = x_weight / norm

        x_score = x_block @ x_weight

        y_loading = (
            y_block.T @ x_score
            / (x_score.T @ x_score)
        )

        updated = (
            y_block @ y_loading
            / (y_loading.T @ y_loading)
        )

        change = np.linalg.norm(updated - y_score)

        scale = max(1.0, np.linalg.norm(y_score))

        y_score = updated

        if change < tolerance * scale:
            break

    x_loading = (
        x_block.T @ x_score
        / (x_score.T @ x_score)
    )

    return x_weight, x_score, x_loading, y_loading


def fit_vip(scaled_matrix, dummy_y, n_components):
    """
    Calculate VIP scores from an n-component PLS-DA model.
    """

    x_residual = scaled_matrix.copy()
    y_residual = dummy_y.copy()

    weights = []
    scores = []
    y_loadings = []

    for _ in range(n_components):

        w, t, p, c = extract_pls_component(x_residual, y_residual)

        x_residual = x_residual - t @ p.T
        y_residual = y_residual - t @ c.T

        weights.append(w.ravel())
        scores.append(t.ravel())
        y_loadings.append(c.ravel())

    weights = np.array(weights).T
    scores = np.array(scores).T
    y_loadings = np.array(y_loadings).T

    explained = np.array([
        (scores[:, i] @ scores[:, i])
        * (y_loadings[:, i] @ y_loadings[:, i])
        for i in range(n_components)
    ])

    normalised = (
        weights
        / np.linalg.norm(weights, axis=0, keepdims=True)
    )

    contribution = (normalised ** 2) @ explained

    return np.sqrt(
        scaled_matrix.shape[1]
        * contribution
        / explained.sum()
    )

## Step 4. Running one comparison

`analyse` does one comparison at a time, for example scW control vs CSDS. Steps in order:

1. Pick the columns for that tissue and those two groups only, skipping blanks, QC, and the excluded outlier.
2. Build the data matrix with samples as rows and metabolites as columns.
3. For each metabolite, calculate the correlation between its peak area and the group. A correlation near 1 or -1 means the metabolite tracks the group almost perfectly, so I treat `|r|` as **consistency**.
4. Scale the data, build the dummy labels, and fit the model to get VIP. I fit it twice, with 2 components and with 1 component, for the sanity check described below.
5. Run a Welch t-test on every metabolite. Welch is used because I do not want to assume the two groups have equal variance.

The results table then holds, for each metabolite: mean peak area, standard deviation, `|r|`, VIP, p-value, `|t|`, log2 fold change, and ranks for area, VIP and p-value.

The last column, `size_x_consistency`, is `sqrt(SD) * |r|`. This is my check that I understand my own model. With Pareto scaling and one component, VIP should be exactly proportional to that quantity, so the correlation between the two should come out as 1.000. If it ever does not, something in the model code is wrong.

In [ ]:
# ==========================================================================
# 3. ANALYSIS
# ==========================================================================

def analyse(
    tissue,
    case_group,
    control_group="control",
    n_components=N_COMPONENTS,
    excel_path=None
):
    """
    Compute all quantities required for Panel 1.
    """

    table, intensities = load_table(excel_path)

    excluded = EXCLUDED_SAMPLE.get(tissue)

    selected = []

    for column in intensities.columns:

        info = describe_injection(column)

        if info is None:
            continue

        if info[0] != tissue:
            continue

        if info[1] not in (control_group, case_group):
            continue

        if excluded and excluded in column:
            continue

        selected.append(column)

    labels = np.array([
        describe_injection(c)[1]
        for c in selected
    ])

    matrix = intensities[selected].values.T

    is_case = (labels == case_group).astype(float)

    # ------------------------------------------------------------------
    # VIP ingredients
    # ------------------------------------------------------------------

    sd_raw = matrix.std(axis=0, ddof=1)

    correlation = np.array([
        np.corrcoef(matrix[:, j], is_case)[0, 1]
        for j in range(matrix.shape[1])
    ])

    scaled = pareto_scale(matrix)

    dummy = make_dummy_y(labels)

    vip = fit_vip(scaled, dummy, n_components)

    vip_one = fit_vip(scaled, dummy, 1)

    # ------------------------------------------------------------------
    # Univariate statistics
    # ------------------------------------------------------------------

    t_stat, p_value = stats.ttest_ind(
        matrix[is_case == 1],
        matrix[is_case == 0],
        axis=0,
        equal_var=False
    )

    results = pd.DataFrame({

        "Metabolite":
            table["Metabolite"].astype(str).values,

        "Confidence":
            table["Confidence in annotation"].astype(str).values,

        "area": matrix.mean(axis=0),

        "SD": sd_raw,

        "abs_r": np.abs(correlation),

        "VIP": vip,

        "VIP_1comp": vip_one,

        "p_value": p_value,

        "t_stat": np.abs(t_stat),

        "log2FC": np.log2(
            matrix[is_case == 1].mean(axis=0)
            / matrix[is_case == 0].mean(axis=0)
        ),

        "size_x_consistency":
            np.sqrt(sd_raw) * np.abs(correlation),
    })

    results["area_rank"] = (
        results["area"].rank(ascending=False, method="min").astype(int)
    )

    results["vip_rank"] = (
        results["VIP"].rank(ascending=False, method="min").astype(int)
    )

    results["p_rank"] = (
        results["p_value"].rank(method="min").astype(int)
    )

    return {
        "results": results,
        "matrix": matrix,
        "labels": labels,
        "tissue": tissue,
        "case_group": case_group,
        "control_group": control_group,
        "n_samples": len(selected),
        "n_components": n_components,
        "metabolite_names": table["Metabolite"].astype(str).values,
    }

## Step 5. Turning p = 0.05 into a line I can draw

The figure has `|r|` on the y-axis, not a p-value, so I cannot just draw a line at 0.05. This function converts the significance threshold into the correlation that matches it.

The idea is simple. For a given number of samples there is a critical t-value for p = 0.05, and there is a fixed formula linking t to r. So for n = 12 samples the answer is about `|r| = 0.576`. Anything above that line is significant, anything below is not.

One caveat I should keep in mind: this threshold comes from a Pearson correlation test, while the p-values in the table come from a Welch t-test. They agree closely but not perfectly, so a couple of points can sit on the wrong side of the line by a hair.

In [ ]:

# ==========================================================================
# 4. SIGNIFICANCE THRESHOLD
# ==========================================================================

def significance_correlation(n_samples, alpha=0.05):
    """
    Absolute correlation corresponding to p = alpha, two-sided.
    """

    degrees_of_freedom = n_samples - 2

    t_critical = stats.t.ppf(1 - alpha / 2, degrees_of_freedom)

    return (
        t_critical
        / np.sqrt(t_critical ** 2 + degrees_of_freedom)
    )


## Step 6. Picking example metabolites to label

I do not want 301 labels on the plot. I want a few examples that show the pattern, so this function picks one metabolite for each type:

* **A. high abundance + high VIP** — abundant, significant, and the highest VIP of that set.
* **B. low abundance + high consistency** — not in the top 60 by area, but tracks the groups very tightly.
* **C. high abundance + high VIP, not significant** — big and influential in the model, yet the t-test is not convinced.
* **D. low VIP + significant** — the model rates it as unimportant, but the p-value is tiny.

C and D are the interesting ones. They are the cases where VIP and the p-value disagree, which is the whole point of the figure.

In [ ]:
# ==========================================================================
# 5. ARCHETYPES
# ==========================================================================

def pick_archetypes(results, abundant_top=ABUNDANT_TOP):
    """
    Select representative metabolites for annotation.
    """

    abundant = results["area_rank"] <= abundant_top

    high_vip = results["VIP"] > 1

    significant = results["p_value"] < 0.05

    def best(subset, column, largest=True):

        if len(subset) == 0:
            return None

        if largest:
            picked = subset.nlargest(1, column)
        else:
            picked = subset.nsmallest(1, column)

        return picked.iloc[0]

    return {

        # The second element is the short descriptor printed under the
        # metabolite name on the figure.

        "A": (
            best(results[high_vip & significant & abundant], "VIP"),
            "high abundance + high VIP"
        ),

        "B": (
            best(results[high_vip & ~abundant], "abs_r"),
            "low abundance + high consistency"
        ),

        "C": (
            best(results[high_vip & ~significant], "VIP"),
            "high abundance + high VIP, not significant"
        )
    }

## Step 7. Deciding the plot window before drawing

This was a bug fix. Originally the labels and curves were placed before the axis limits were known, so some of them ended up outside the frame and the saved figures had odd empty bands.

`compute_limits` now works out the visible window first:

* the y-axis runs from just below the significance line up to the highest visible point,
* the x-axis covers only the points that are actually shown, plus a little padding on each side.

`axes_fraction` is a small helper that converts a data point into "how far across and how far up the plot is this, from 0 to 1". Since the x-axis is on a log scale, it takes the log before doing the maths. I use it later to decide which side of a point its label should go on.

In [ ]:
# ==========================================================================
# 6. AXIS LIMITS
# ==========================================================================

def compute_limits(results, threshold):
    """
    Work out the visible window BEFORE anything is drawn, so that every
    later element (curves, labels, annotations) can be aligned to it.
    """

    visible = results[
        (results["abs_r"] >= threshold)
        & (results["area"] > 0)
        & np.isfinite(results["area"])
    ]

    if len(visible) == 0:
        visible = results[results["area"] > 0]

    top = float(visible["abs_r"].max())

    span = max(top - threshold, 0.05)

    y_min = threshold - BOTTOM_MARGIN * span
    y_max = min(1.0, top + 0.08 * span)

    log_min = np.log10(visible["area"].min())
    log_max = np.log10(visible["area"].max())

    x_min = 10 ** (log_min - X_PAD_DECADES_LEFT)
    x_max = 10 ** (log_max + X_PAD_DECADES_RIGHT)

    return (x_min, x_max), (y_min, y_max)


def axes_fraction(axis, x, y):
    """
    Position of a data point as a (0-1, 0-1) fraction of the axes,
    accounting for the log x-scale.
    """

    x_min, x_max = axis.get_xlim()
    y_min, y_max = axis.get_ylim()

    fx = (
        (np.log10(x) - np.log10(x_min))
        / (np.log10(x_max) - np.log10(x_min))
    )

    fy = (y - y_min) / (y_max - y_min)

    return fx, fy

## Step 8. The dotted equal-VIP curves

These dotted lines show where VIP is roughly 0.5, 1 and 3. They are the visual way of saying that VIP depends on two things at once: how abundant a metabolite is, and how consistently it tracks the group.

How it works:

1. Abundance and spread go together, so I fit a straight line through `log10(area)` against `log10(sqrt(SD))` and use it to predict the spread at any abundance.
2. I work out the constant that links `sqrt(SD) * |r|` to VIP, using the median across all metabolites.
3. For each level I solve for the `|r|` that would give that VIP at each abundance, and draw the result.

The curves slope downwards. That means an abundant metabolite only needs a modest correlation to reach VIP = 1, while a rare metabolite needs a very strong correlation to get there. This is a property of the method, not biology, and it is worth saying out loud when presenting the figure.

Everything here is clipped to the axes and the labels are anchored just inside the top edge, so nothing spills outside the frame.

In [ ]:
# ==========================================================================
# 7. VIP CURVES
# ==========================================================================

def draw_equal_vip_curves(axis, results, levels=VIP_CURVE_LEVELS):
    """
    Draw dotted curves of equal VIP.

    Both curves and their labels are clipped to the current axes window,
    so nothing is drawn above the frame.
    """

    usable = results[
        (results["area"] > 0)
        & (results["SD"] > 0)
        & np.isfinite(results["VIP"])
        & np.isfinite(results["abs_r"])
    ]

    if len(usable) < 10:
        return

    slope, intercept = np.polyfit(
        np.log10(usable["area"]),
        np.log10(np.sqrt(usable["SD"])),
        1
    )

    x_min, x_max = axis.get_xlim()
    y_min, y_max = axis.get_ylim()

    x_values = np.logspace(
        np.log10(x_min),
        np.log10(x_max),
        400
    )

    predicted_sqrt_sd = 10 ** (
        intercept + slope * np.log10(x_values)
    )

    product = (
        np.sqrt(usable["SD"].values)
        * usable["abs_r"].values
    )

    good = (product > 0) & (usable["VIP"].values > 0)

    if not good.any():
        return

    constant = np.median(
        usable["VIP"].values[good] / product[good]
    )

    # Keep the labels off the top frame by a fixed fraction of the y-range.
    label_gap = 0.015 * (y_max - y_min)

    for level in levels:

        curve = level / (constant * predicted_sqrt_sd)

        inside = (curve >= y_min) & (curve <= y_max)

        if inside.sum() < 3:
            continue

        axis.plot(
            x_values[inside],
            curve[inside],
            linestyle=":",
            linewidth=1.0,
            color="black",
            alpha=0.6,
            zorder=1,
            clip_on=True
        )

        # Anchor the label to the highest visible point of the curve
        # and hang it just below the top of the plot.
        highest = int(np.argmax(np.where(inside, curve, -np.inf)))

        label_x = x_values[highest]
        label_y = min(curve[highest], y_max - label_gap)

        fx, _ = axes_fraction(axis, label_x, label_y)

        if fx < 0.06:
            alignment = "left"
        elif fx > 0.94:
            alignment = "right"
        else:
            alignment = "center"

        axis.text(
            label_x,
            label_y,
            f"VIP ≈ {level:g}",
            fontsize=7,
            alpha=0.85,
            va="top",
            ha=alignment,
            zorder=5,
            clip_on=True,
            bbox=dict(
                facecolor="white",
                edgecolor="none",
                alpha=0.75,
                pad=1.0
            )
        )


## Step 9. Drawing the panel

The order of the drawing matters, so the cell follows it strictly:

1. Set the log x-scale and fix both axis limits.
2. Draw the scatter. Each point is one metabolite. Position is abundance and consistency, and **both size and colour show VIP**, so high-VIP metabolites are big and yellow.
3. Draw the equal-VIP curves, which now know the window they have to fit into.
4. Draw the dashed p = 0.05 line and label it.
5. Add the labels for the example metabolites from Step 6. Each one gets the name in bold and the category underneath in italics. If a point sits near the right edge the label flips to the left, so it never runs under the colour bar.

Points below the significance line are outside the window, so only significant metabolites are visible.

One thing to fix before this goes in a report: the y-axis label says p-value on a log scale, but the values being plotted are `|r|` on a linear scale. The label needs to match the data, or the data needs to change to p-values.

In [ ]:
# ==========================================================================
# 8. PANEL 1
# ==========================================================================

def panel_map(axis, analysis, archetypes):
    """
    Panel 1.

    X: peak area (log scale)
    Y: p value

    Limits are fixed first; every other element is then positioned
    relative to those limits.
    """

    results = analysis["results"]

    threshold = significance_correlation(analysis["n_samples"])

    # ------------------------------------------------------------------
    # 1. Scale and limits FIRST
    # ------------------------------------------------------------------

    axis.set_xscale("log")

    (x_min, x_max), (y_min, y_max) = compute_limits(results, threshold)

    axis.set_xlim(x_min, x_max)
    axis.set_ylim(y_min, y_max)

    # ------------------------------------------------------------------
    # 2. Scatter
    # ------------------------------------------------------------------

    scatter = axis.scatter(
        results["area"],
        results["abs_r"],
        s=np.clip(results["VIP"] * 22, 4, 260),
        c=np.clip(results["VIP"], 0, VIP_COLOUR_CAP),
        cmap="viridis",
        vmin=0,
        vmax=VIP_COLOUR_CAP,
        alpha=0.78,
        edgecolors="none",
        zorder=3,
        clip_on=True
    )

    # ------------------------------------------------------------------
    # 3. Equal-VIP curves (aligned to the limits set above)
    # ------------------------------------------------------------------

    draw_equal_vip_curves(axis, results)

    # ------------------------------------------------------------------
    # 4. Significance line and its label
    # ------------------------------------------------------------------

    axis.axhline(
        threshold,
        linestyle="--",
        color="grey",
        linewidth=1.2,
        zorder=2
    )

    axis.annotate(
        "p < 0.05",
        xy=(0.012, threshold),
        xycoords=("axes fraction", "data"),
        xytext=(0, 4),
        textcoords="offset points",
        fontsize=9,
        color="dimgrey",
        ha="left",
        va="bottom",
        zorder=6,
        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.85,
            pad=1.2
        )
    )

    # ------------------------------------------------------------------
    # 5. Archetype annotations, placed away from the frame
    # ------------------------------------------------------------------

    label_box = dict(
        facecolor="white",
        edgecolor="none",
        alpha=0.75,
        pad=1.5
    )

    for _key, (row, descriptor) in archetypes.items():

        if row is None:
            continue

        if not (y_min <= row["abs_r"] <= y_max):
            continue

        if not (x_min <= row["area"] <= x_max):
            continue

        fx, fy = axes_fraction(axis, row["area"], row["abs_r"])

        # Point the label inwards when the marker sits near an edge,
        # so text never runs under the colour bar or off the frame.
        if fx > 0.55:
            dx, ha = -18, "right"
        else:
            dx, ha = 18, "left"

        if fy > 0.80:
            dy, va = -14, "top"
        elif fy < 0.20:
            dy, va = 14, "bottom"
        else:
            dy, va = 11, "bottom"

        name = str(row["Metabolite"])[:26]

        # The two lines are drawn separately so that the metabolite name
        # can be bold while the descriptor stays light. Both use the same
        # offset anchor, so they stay left/right aligned with each other.
        line_gap = 9.5

        if va == "bottom":
            # The block grows upwards: descriptor sits on the inner line.
            name_offset = (dx, dy + line_gap)
            descriptor_offset = (dx, dy)
            anchored = "descriptor"
        else:
            # The block grows downwards: the name sits on the inner line.
            name_offset = (dx, dy)
            descriptor_offset = (dx, dy - line_gap)
            anchored = "name"

        arrow = dict(
            arrowstyle="-",
            lw=0.6,
            color="grey",
            shrinkA=2,
            shrinkB=3
        )

        axis.annotate(
            name,
            xy=(row["area"], row["abs_r"]),
            textcoords="offset points",
            xytext=name_offset,
            fontsize=8,
            fontweight="bold",
            ha=ha,
            va=va,
            zorder=7,
            annotation_clip=True,
            bbox=label_box,
            arrowprops=arrow if anchored == "name" else None
        )

        axis.annotate(
            descriptor,
            xy=(row["area"], row["abs_r"]),
            textcoords="offset points",
            xytext=descriptor_offset,
            fontsize=7,
            style="italic",
            color="#444444",
            ha=ha,
            va=va,
            zorder=7,
            annotation_clip=True,
            bbox=label_box,
            arrowprops=arrow if anchored == "descriptor" else None
        )

    # ------------------------------------------------------------------
    # 6. Labels
    # ------------------------------------------------------------------

    axis.set_xlabel("Peak area (abundance) - log scale")
    axis.set_ylabel("p-value (Welch) - log scale, most significant at top")

    axis.set_title(
        "VIP depends on abundance and consistency: only significant metabolites are displayed (Welch p < 0.05)",
        fontsize=11,
        pad=8
    )

    return scatter

## Step 10. Assembling the figure

`build_figure` puts the panel, the colour bar and the titles together.

`layout="constrained"` lets matplotlib work out the spacing, so all four figures come out with the axes in the same place. I also removed `bbox_inches="tight"` from the save call on purpose. That option crops to whatever has been drawn, so a single stray label used to change the size of the whole image and make the panels impossible to compare side by side.

In [ ]:
# ==========================================================================
# 9. FIGURE
# ==========================================================================

def build_figure(analysis, save_as=None):
    """
    Build Panel 1 with a constrained layout so that the title, the axes
    and the colour bar all line up the same way in every comparison.
    """

    results = analysis["results"]

    archetypes = pick_archetypes(results)

    figure, axis = plt.subplots(
        figsize=(10, 7),
        layout="constrained"
    )

    scatter = panel_map(axis, analysis, archetypes)

    colour_bar = figure.colorbar(
        scatter,
        ax=axis,
        fraction=0.046,
        pad=0.02
    )

    colour_bar.set_label(f"VIP")

    figure.suptitle(
        f"{analysis['tissue']}: "
        f"{analysis['control_group']} vs "
        f"{analysis['case_group']} "
        f"({len(results)} annotated metabolites)",
        fontsize=13
    )

    if save_as:

        path = os.path.join(OUTPUT_DIR, save_as)

        # No bbox_inches="tight": the figure size is now fixed and every
        # element lives inside the axes, so all four panels come out
        # with identical proportions.
        figure.savefig(path, dpi=200, facecolor="white")

        plt.close(figure)

        return path

    return figure

## Step 11. Running all four comparisons

`main` loops over the four comparisons: scW vs CMVS, scW vs CSDS, BAT vs CMVS, BAT vs CSDS. For each one it runs the analysis, prints a few diagnostics, and saves the figure.

What the printed numbers mean:

* `n` is the number of samples in that comparison, which is small. Everything else should be read with that in mind.
* `significance threshold |r|` is the height of the dashed line.
* `VIP (1 comp) vs sqrt(SD)x|r|` should be 1.0000. It is the check from Step 4 that the model is behaving as expected.
* `|t| vs sqrt(SD)` is close to 0, so the t-statistic barely cares about how abundant a metabolite is.
* `|t| vs |r|` is close to 1, because the t-test and the correlation are measuring nearly the same thing.

Those last two lines are the short version of the whole story. The t-test responds to consistency alone, while VIP responds to consistency and abundance together.

In [ ]:
# ==========================================================================
# 10. MAIN
# ==========================================================================

def main():

    comparisons = [
        ("scW", "CMVS"),
        ("scW", "CSDS"),
        ("BAT", "CMVS"),
        ("BAT", "CSDS"),
    ]

    for tissue, case in comparisons:

        analysis = analyse(tissue, case)

        results = analysis["results"]

        identity_r = np.corrcoef(
            results["VIP_1comp"],
            results["size_x_consistency"]
        )[0, 1]

        size_r = np.corrcoef(
            results["t_stat"],
            np.sqrt(results["SD"])
        )[0, 1]

        consistency_r = np.corrcoef(
            results["t_stat"],
            results["abs_r"]
        )[0, 1]

        threshold = significance_correlation(analysis["n_samples"])

        print(f"\n{tissue}: {analysis['control_group']} vs {case}")
        print(f"  n = {analysis['n_samples']}")
        print(f"  components = {analysis['n_components']}")
        print(f"  significance threshold |r| = {threshold:.4f}")
        print(f"  VIP (1 comp) vs sqrt(SD)x|r| : r = {identity_r:.4f}")
        print(f"  |t| vs sqrt(SD) : r = {size_r:.3f}")
        print(f"  |t| vs |r| : r = {consistency_r:.3f}")

        path = build_figure(
            analysis,
            save_as=f"panel1_vip_vs_pvalue_{tissue}_{case}.png"
        )

        print(f"  -> {path}")

## Step 12. Go

This runs everything. Four PNG files appear in `OUTPUT_DIR`, one per comparison.

In [ ]:

# ==========================================================================
# RUN
# ==========================================================================

if __name__ == "__main__":
    main()


scW: control vs CMVS
  n = 12
  components = 2
  significance threshold |r| = 0.5760
  VIP (1 comp) vs sqrt(SD)x|r| : r = 1.0000
  |t| vs sqrt(SD) : r = 0.065
  |t| vs |r| : r = 0.949
  -> ./panel1_vip_vs_pvalue_scW_CMVS.png

scW: control vs CSDS
  n = 12
  components = 2
  significance threshold |r| = 0.5760
  VIP (1 comp) vs sqrt(SD)x|r| : r = 1.0000
  |t| vs sqrt(SD) : r = 0.047
  |t| vs |r| : r = 0.970
  -> ./panel1_vip_vs_pvalue_scW_CSDS.png

BAT: control vs CMVS
  n = 11
  components = 2
  significance threshold |r| = 0.6021
  VIP (1 comp) vs sqrt(SD)x|r| : r = 1.0000
  |t| vs sqrt(SD) : r = 0.134
  |t| vs |r| : r = 0.954
  -> ./panel1_vip_vs_pvalue_BAT_CMVS.png

BAT: control vs CSDS
  n = 12
  components = 2
  significance threshold |r| = 0.5760
  VIP (1 comp) vs sqrt(SD)x|r| : r = 1.0000
  |t| vs sqrt(SD) : r = 0.013
  |t| vs |r| : r = 0.985
  -> ./panel1_vip_vs_pvalue_BAT_CSDS.png


---

## How to read the figures

Each point is one metabolite.

* **Right** means abundant. **Up** means it tracks the stress group consistently.
* **Big and yellow** means high VIP. **Small and purple** means low VIP.
* The dashed line at the bottom is p = 0.05. Only significant metabolites are inside the window.
* The dotted curves are levels of equal VIP.

The message: the big yellow points sit on the right, not at the top. A metabolite can get a high VIP mostly by being abundant, and a metabolite can have a tiny p-value while barely registering in the model.

## Things I still need to sort out

1. **The y-axis label does not match the data.** It says p-value on a log scale, but `|r|` on a linear scale is what is plotted.
2. **The title says gene expression** and the data is metabolomics.
3. **No multiple-testing correction.** With 301 metabolites at p < 0.05, roughly 15 hits are expected by chance alone. I should add a Benjamini-Hochberg FDR column before calling anything a finding.
4. **Small n.** Around 5 to 6 animals per group. Both the p-values and the VIP scores are unstable at that size, and the model has not been cross-validated or permutation-tested, so I cannot say yet whether the separation is real.
5. **One sample was dropped** from BAT (`BAT_Aq_CMVS6`). That decision needs to be written up properly, with the PCA that justified it.
6. **The significance line and the p-values come from two different tests** (Pearson vs Welch), so a few points sit on the wrong side of the line. Worth making them consistent.

## Next steps

* Add FDR-adjusted p-values and mark them on the figure.
* Cross-validate the PLS-DA model (Q2) and run a permutation test.
* Compare the metabolites that come out on top in both tissues, and check whether CSDS and CMVS point in the same direction.
* Map the strongest hits onto pathways.